# Setup-Check

Dieses Notebook prüft, ob deine Umgebung für das Modul bereit ist.

1. Wähle oben rechts den Kernel mit deiner installierten Python-Version aus.
2. Führe alle Zellen von oben nach unten aus (`Shift+Enter` oder `Run All`).
3. Am Ende steht entweder **Alles bereit** oder eine Liste mit Problemen und was du dagegen tun kannst.

Die Anleitung dazu findest du in der Wissensdatenbank unter
[Python Installation](https://abbts-dat-ski.github.io/wissensdatenbank/python_installation/).

## 1. Packages installieren

Diese Zelle installiert alle Packages für das Semester genau in die Python-Version, die dieses Notebook verwendet.
Das dauert beim ersten Mal ein paar Minuten. Wenn alles schon installiert ist, geht es schnell.

Diese Zeile ist Code für das Notebook. Führe sie hier als Zelle aus und nicht im Terminal.

In [ ]:
%pip install jupyter pandas plotly nbformat matplotlib scikit-learn

## 2. Umgebung prüfen

In [ ]:
import importlib
import os
import sys
from pathlib import Path

problems = []


def ok(message):
    print("[OK]     " + message)


def problem(message, fix):
    print("[FEHLER] " + message)
    print("         Lösung: " + fix)
    problems.append(message)


# Python-Version
version = sys.version_info
version_text = f"{version.major}.{version.minor}.{version.micro}"
if (version.major, version.minor) < (3, 12):
    problem(
        f"Python {version_text} ist zu alt.",
        "Installiere Python 3.13 von python.org und wähle danach den neuen Kernel aus.",
    )
else:
    ok(f"Python {version_text}")

# Microsoft-Store-Python macht oft Probleme mit Pfaden und Berechtigungen
if "WindowsApps" in sys.executable:
    print("[HINWEIS] Python aus dem Microsoft Store wird verwendet. Das geht, solange alles unten OK ist.")
    print("          Falls Probleme auftauchen: Python 3.13 von python.org installieren und diesen Kernel wählen.")

# Packages
for package in ["pandas", "plotly", "nbformat", "matplotlib", "sklearn"]:
    try:
        module = importlib.import_module(package)
        ok(f"{package} {getattr(module, '__version__', '')}")
    except ImportError:
        fix = (
            "Führe die Zelle unter '1. Packages installieren' nochmals aus. "
            "Starte danach oben über 'Restart' den Kernel neu und führe dieses Notebook erneut aus."
        )
        if (version.major, version.minor) > (3, 14):
            fix += (
                f" Python {version.major}.{version.minor} ist sehr neu und noch nicht alle Packages unterstützen sie."
                " Installiere in diesem Fall Python 3.13 von python.org und wähle diesen Kernel aus."
            )
        problem(f"{package} fehlt.", fix)

# Ordnerstruktur: ../data muss neben dem Unterrichtsblock-Ordner liegen
cwd = Path.cwd()
data_dir = cwd.parent / "data"
print(f"         Notebook-Ordner: {cwd}")
if cwd.name == cwd.parent.name:
    problem(
        f"Der Ordner '{cwd.name}' wurde doppelt entpackt.",
        f"Verschiebe den inneren Ordner '{cwd.name}' eine Ebene nach oben, direkt neben 'data'.",
    )
uses_git = (cwd.parent / ".git").is_dir()
print(f"         Material: {'Git (neue Dateien mit git pull)' if uses_git else 'ZIP-Dateien'}")
if not data_dir.is_dir():
    problem(
        "Der Ordner '../data' wurde nicht gefunden.",
        r"Mit Git: klone das Repository direkt nach C:\DAT-SKI (Python Installation, Abschnitt 2). "
        "Mit ZIP: entpacke data.zip in denselben Oberordner wie diesen Unterrichtsblock-Ordner, "
        r"zum Beispiel C:\DAT-SKI\data und C:\DAT-SKI\Unterrichtsblock-1.",
    )
elif not (data_dir / "data_resampled.csv").is_file():
    problem(
        "Im Ordner '../data' fehlen Dateien.",
        r"Führe im Ordner C:\DAT-SKI den Befehl git pull aus."
        if uses_git
        else "Lade data.zip neu herunter und entpacke es vollständig, nicht nur einzelne Dateien.",
    )
else:
    ok(f"Datenordner gefunden: {data_dir.resolve()}")

if "OneDrive" in str(cwd):
    problem(
        "Der Arbeitsordner liegt in OneDrive.",
        r"Verschiebe den ganzen Ordner an einen kurzen Pfad wie C:\DAT-SKI\.",
    )
if len(str(cwd)) > 120:
    problem(
        "Der Pfad zum Arbeitsordner ist sehr lang.",
        r"Verschiebe den ganzen Ordner an einen kurzen Pfad wie C:\DAT-SKI\.",
    )

## 3. Daten einlesen und plotten

Wenn alles funktioniert, siehst du unten eine kleine Tabelle und ein interaktives Diagramm.

In [ ]:
try:
    import pandas as pd
    import plotly.express as px

    if not (data_dir / "data_resampled.csv").is_file():
        print("[--]     Übersprungen, weil die Daten fehlen (siehe oben).")
    else:
        df = pd.read_csv("../data/data_resampled.csv", parse_dates=["timestamp"])
        ok(f"Daten eingelesen: {len(df)} Zeilen")
        room = df[(df["room_id"] == 415) & (df["co2"] > 0)]
        fig = px.line(room, x="timestamp", y="co2", title="Setup-Check: CO2 in Raum 415")
        fig.show()
        display(df.head(3))
except Exception as error:
    problem(
        f"Daten konnten nicht eingelesen oder geplottet werden ({type(error).__name__}: {error}).",
        "Behebe zuerst die Fehler weiter oben und führe danach alle Zellen erneut aus.",
    )

## 4. Ergebnis

In [ ]:
print()
if problems:
    print(f"Es gibt noch {len(problems)} Problem(e):")
    for message in problems:
        print(" - " + message)
    print("\nBehebe die Punkte von oben nach unten und führe danach alle Zellen erneut aus.")
    print("Falls du nicht weiterkommst: Mach einen Screenshot dieser Ausgabe und zeige ihn im Unterricht.")
else:
    print("Alles bereit. Dein Setup funktioniert für das ganze Semester.")